In [2]:
# ═══════════════════════════════════════════════════════════════════════════════
# NOTEBOOK 03 — DiCE Counterfactual Explanations   (REVISED, STAGE-1 DIAGNOSTIC)
# ═══════════════════════════════════════════════════════════════════════════════
# Self-contained: loads only primitives from Notebook 01 (df_final, best_models,
# config). Reproduces the original random-method DiCE recourse analysis, but adds
# the diagnostics needed to answer the referees before committing to the heavier
# genetic-method run (Stage 2):
#
#   [ACT]   Actionability split — only clinically modifiable lifestyle/diet
#           features are allowed to vary; structural variables (income, education,
#           household income, health-screening) are held immutable. Answers the
#           referee question "what is actionable?" and removes nonsensical recourse
#           (e.g. "raise your income to lower glucose").
#   [MAHA]  Mahalanobis proximity reported ALONGSIDE the original Euclidean
#           proximity. Euclidean treats features as independent; Mahalanobis
#           respects their covariance (waist↔BMI↔weight, smoking↔lipids). A large
#           Euclidean/Mahalanobis gap quantifies how far the independence
#           assumption distorts the recourse-burden estimate (referee comment).
#   [CI]    Bootstrap confidence intervals on group proximity AND on the
#           elderly/young proximity ratio — tests whether the headline "9.4×"
#           survives the small (3-case) sample or is noise (referee: uncertainty).
#   [ABSENT] Recourse-absence is tracked explicitly: for every representative
#           case we record whether ANY actionable counterfactual exists. When
#           structural variables are held immutable, some groups (notably Elderly
#           Male IFG→Normal) have NO feasible recourse at all — a stronger and more
#           honest finding than a large-but-finite "9.4x burden". This is the core
#           Stage-1 result: recourse ABSENCE, not merely recourse difficulty.
#   [FRAME] Proximity is reframed as a DIAGNOSTIC distance-to-decision-boundary,
#           NOT a causal lifestyle prescription (cross-sectional data).
#   [PATHS/FIG] results/{tables,figures,artifacts}; seaborn grayscale; no
#           captions; dpi=600; png + pdf; no absolute paths printed.
#
# NOTE: method="random" is retained for Stage 1 (fast). It has no explicit
#       proximity/diversity loss; Stage 2 will switch to method="genetic" (which
#       optimises an explicit objective) once this diagnostic confirms direction.
# ═══════════════════════════════════════════════════════════════════════════════

import warnings; warnings.filterwarnings("ignore")
import logging
from pathlib import Path

import numpy as np
import pandas as pd
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

import dice_ml
from dice_ml import Dice
from sklearn.model_selection import train_test_split

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s")
log = logging.getLogger(__name__)

SEED = 42; DPI = 600
N_CF = 3          # counterfactuals per case
N_CASES = 3       # representative cases per group × transition (Stage-1 setting)
N_BOOT = 2000     # bootstrap resamples for CIs
np.random.seed(SEED)

sns.set_theme(style="whitegrid", context="paper")
sns.set_palette("Greys")
plt.rcParams.update({"font.family":"DejaVu Sans", "font.size":11,
                     "axes.unicode_minus":False, "figure.dpi":150, "savefig.dpi":DPI,
                     "axes.edgecolor":"0.2", "grid.color":"0.85"})

def save_fig(fig, name):
    for ext in ("png", "pdf"):
        fig.savefig(FIG_DIR / f"{name}.{ext}", dpi=DPI, bbox_inches="tight")
    plt.close(fig)

def find_project_root(start: Path = Path.cwd()) -> Path:
    for p in [start, *start.parents]:
        if (p / "data").is_dir() and (p / "results").is_dir():
            return p
    return start.parent if start.name == "notebooks" else start

ROOT      = find_project_root()
FIG_DIR   = ROOT / "results" / "figures"
TABLE_DIR = ROOT / "results" / "tables"
ART_DIR   = ROOT / "results" / "artifacts"
for d in (FIG_DIR, TABLE_DIR, ART_DIR):
    d.mkdir(parents=True, exist_ok=True)

# ═══════════════════════════════════════════════════════════════════════════════
# 1. LOAD PRIMITIVES FROM NOTEBOOK 01
# ═══════════════════════════════════════════════════════════════════════════════
def _load_df_final():
    pq = ART_DIR / "df_final.parquet"; cs = ART_DIR / "df_final.csv"
    if pq.exists():
        try: return pd.read_parquet(pq)
        except Exception: pass
    return pd.read_csv(cs)

df_final    = _load_df_final()
best_models = joblib.load(ART_DIR / "best_models.pkl")
best_algo_name = joblib.load(ART_DIR / "best_algo_name.pkl")
cfg         = joblib.load(ART_DIR / "config.pkl")

X_FEATURES   = cfg["X_FEATURES"]; NUM_FEATURES = cfg["NUM_FEATURES"]
CAT_FEATURES = cfg["CAT_FEATURES"]
GROUP_CONFIG = cfg["GROUP_CONFIG"]; GROUP_LABELS = cfg["GROUP_LABELS"]

# Coerce numeric (CSV round-trip safety)
_numc = [c for c in (X_FEATURES + ["FPG","AgeGroup","Sex","SurveyYear"]) if c in df_final.columns]
df_final[_numc] = df_final[_numc].apply(pd.to_numeric, errors="coerce").fillna(0.0).astype("float64")

def assign_glucose_stage(fpg):
    if fpg < 100:  return "normal"
    if fpg < 126:  return "ifg"
    return "diabetes"

TRANSITIONS = {
    "diabetes": {"target_range":(100.0,125.0), "label":"Diabetes → IFG",
                 "source_label":"Diabetes (>=126)", "target_label":"IFG (100-125)"},
    "ifg":      {"target_range":(75.0,99.9),   "label":"IFG → Normal",
                 "source_label":"IFG (100-125)",  "target_label":"Normal (<100)"},
}

# ── [ACT] Actionability split ─────────────────────────────────────────────────
# Structural / non-behavioural variables are held IMMUTABLE; only clinically
# modifiable lifestyle & diet variables may vary in a counterfactual.
IMMUTABLE = ["IncomeQuartile", "HouseholdIncome", "EducationLevel", "HealthScreening"]
FEATURES_TO_VARY = [f for f in X_FEATURES if f not in IMMUTABLE]
log.info("Actionable features: %d / %d (immutable: %s)",
         len(FEATURES_TO_VARY), len(X_FEATURES), ", ".join(IMMUTABLE))
log.info("Primitives loaded. df_final: %s", df_final.shape)

# ═══════════════════════════════════════════════════════════════════════════════
# 2. HELPERS — DiCE + proximity (Euclidean & Mahalanobis) + bootstrap CI
# ═══════════════════════════════════════════════════════════════════════════════
def force_float(df, features):
    df = df.copy()
    for c in features:
        df[c] = pd.to_numeric(df[c], errors="coerce").astype("float64")
    return df

def build_dice_explainer(model, df_train, feature_names, outcome="FPG"):
    d = dice_ml.Data(dataframe=df_train[feature_names+[outcome]],
                     continuous_features=feature_names, outcome_name=outcome)
    m = dice_ml.Model(model=model, backend="sklearn", model_type="regressor")
    return Dice(d, m, method="random")

def generate_counterfactuals(explainer, query, target_range, n_cf=N_CF):
    query = force_float(query, query.columns.tolist())
    try:
        return explainer.generate_counterfactuals(
            query_instances=query, total_CFs=n_cf,
            desired_range=list(target_range), permitted_range=None,
            features_to_vary=FEATURES_TO_VARY,   # [ACT] only actionable features
            random_seed=SEED)
    except Exception as exc:
        log.warning("CF generation failed: %s", exc)
        return None

def feature_stds(df_train, features):
    s = df_train[features].std().values.astype(float)
    s[s == 0] = 1.0
    return s

def inv_cov(df_train, features):
    """Pseudo-inverse covariance for Mahalanobis (respects feature correlation)."""
    C = np.cov(df_train[features].values.astype(float).T)
    C += np.eye(C.shape[0]) * 1e-6          # ridge for numerical stability
    return np.linalg.pinv(C)

def proximity_euclid(orig, cf_df, features, stds):
    o = orig[features].values.flatten().astype(float)
    sc = [float(np.mean(np.abs(o - r.values.astype(float))/stds))
          for _, r in cf_df[features].iterrows()]
    return round(float(np.mean(sc)), 4)

def proximity_maha(orig, cf_df, features, VI):
    o = orig[features].values.flatten().astype(float)
    sc = []
    for _, r in cf_df[features].iterrows():
        d = o - r.values.astype(float)
        sc.append(float(np.sqrt(max(d @ VI @ d, 0.0))))
    return round(float(np.mean(sc)), 4)

def diversity(cf_df, features, stds):
    if len(cf_df) < 2: return 0.0
    v = cf_df[features].values.astype(float)
    dd = [float(np.mean(np.abs(v[i]-v[j])/stds))
          for i in range(len(v)) for j in range(i+1,len(v))]
    return round(float(np.mean(dd)), 4)

def feasibility(orig, cf_df, cat_features):
    oc = orig[cat_features].values.flatten().astype(float)
    n = len(cf_df)*len(cat_features)
    if n == 0: return 0.0
    valid = sum(int(abs(cv-ov)<=1.0)
                for _, r in cf_df[cat_features].iterrows()
                for ov, cv in zip(oc, r.values.astype(float)))
    return round(valid/n, 4)

def sparsity(orig, cf_df, features, tol=1e-3):
    o = orig[features].values.flatten().astype(float)
    sc = [np.sum(np.abs(o - r.values.astype(float))>tol)/len(features)
          for _, r in cf_df[features].iterrows()]
    return round(float(np.mean(sc)), 4)

def bootstrap_ci(values, n_boot=N_BOOT, ci=0.95, seed=SEED):
    v = np.asarray(values, dtype=float)
    if len(v) < 2: return (np.nan, np.nan)
    rng = np.random.RandomState(seed)
    boots = [np.mean(rng.choice(v, len(v), replace=True)) for _ in range(n_boot)]
    return (round(float(np.percentile(boots,(1-ci)/2*100)),4),
            round(float(np.percentile(boots,(1+ci)/2*100)),4))

log.info("DiCE + proximity + bootstrap utilities defined.")

# ═══════════════════════════════════════════════════════════════════════════════
# 3. STRATIFIED DiCE GENERATION  (random method; actionable-only)
# ═══════════════════════════════════════════════════════════════════════════════
all_cf_results = {}; quality_records = []; recourse_records = []   # [ABSENT]

for grp, gcfg in GROUP_CONFIG.items():
    log.info("── Group: %s ──", GROUP_LABELS[grp])
    all_cf_results[grp] = {}
    df_g = df_final[(df_final["AgeGroup"]==gcfg["age_group"]) &
                    (df_final["Sex"]==gcfg["sex_code"])].copy().reset_index(drop=True)
    df_g = force_float(df_g, X_FEATURES)
    df_g["FPG"] = pd.to_numeric(df_g["FPG"], errors="coerce").astype("float64")

    model = best_models.get(grp)
    if model is None:
        log.warning("  No model for %s — skipping", grp); continue

    X, y = df_g[X_FEATURES], df_g["FPG"]
    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.20, random_state=SEED)
    df_train = force_float(X_train.copy(), X_FEATURES); df_train["FPG"] = y_train.values.astype("float64")
    stds = feature_stds(df_train, X_FEATURES)
    VI   = inv_cov(df_train, X_FEATURES)                 # [MAHA]

    try:
        explainer = build_dice_explainer(model, df_train, X_FEATURES)
    except Exception as exc:
        log.error("  DiCE explainer failed for %s: %s", grp, exc); continue

    df_val = force_float(X_val.copy(), X_FEATURES); df_val["FPG"] = y_val.values.astype("float64")
    df_val["Stage"] = df_val["FPG"].apply(assign_glucose_stage)

    for stage, t in TRANSITIONS.items():
        sub = df_val[df_val["Stage"]==stage].reset_index(drop=True)
        if len(sub)==0:
            log.info("  [%s] no hold-out cases for stage '%s'", grp, stage)
            recourse_records.append({"Group":GROUP_LABELS[grp], "_key":grp,
                "Transition":t["label"], "n_eligible":0, "n_attempted":0,
                "n_recourse":0, "n_absent":0})
            continue
        med = sub["FPG"].median()
        rep = sub.iloc[(sub["FPG"]-med).abs().argsort()].head(N_CASES)
        log.info("  [%s → %s] eligible=%d representative=%d",
                 t["source_label"], t["target_label"], len(sub), len(rep))

        case_list=[]
        n_attempted = 0; n_recourse = 0            # [ABSENT] per group×transition
        for ci_, (_, row) in enumerate(rep.iterrows()):
            query = force_float(row[X_FEATURES].to_frame().T.reset_index(drop=True), X_FEATURES)
            orig_fpg = float(row["FPG"])
            n_attempted += 1
            res = generate_counterfactuals(explainer, query, t["target_range"], n_cf=N_CF)
            # [ABSENT] a None result OR empty CF set means NO actionable recourse exists
            if res is None:
                log.info("    Case %d | orig=%.1f | NO RECOURSE (no actionable CF found)",
                         ci_+1, orig_fpg)
                continue
            try:
                cf_df = res.cf_examples_list[0].final_cfs_df
                if cf_df is None or len(cf_df)==0:
                    log.info("    Case %d | orig=%.1f | NO RECOURSE (empty CF set)",
                             ci_+1, orig_fpg)
                    continue
                cf_df = force_float(cf_df, X_FEATURES)
                n_recourse += 1                     # recourse exists for this case
                delta_df = cf_df[X_FEATURES].subtract(query[X_FEATURES].values[0], axis=1)
                prox_e = proximity_euclid(query, cf_df, X_FEATURES, stds)
                prox_m = proximity_maha(query, cf_df, X_FEATURES, VI)     # [MAHA]
                div  = diversity(cf_df, X_FEATURES, stds)
                feas = feasibility(query, cf_df, CAT_FEATURES)
                spar = sparsity(query, cf_df, X_FEATURES)
                pred = model.predict(cf_df[X_FEATURES]).tolist()
                log.info("    Case %d | orig=%.1f ProxE=%.4f ProxM=%.4f Div=%.4f Feas=%.4f Spar=%.4f",
                         ci_+1, orig_fpg, prox_e, prox_m, div, feas, spar)
                case_list.append({"case_idx":ci_+1, "orig_fpg":orig_fpg, "pred_fpg":pred,
                                  "cf_df":cf_df, "delta_df":delta_df, "query":query,
                                  "n_eligible":len(sub)})
                quality_records.append({"Group":GROUP_LABELS[grp], "_key":grp,
                    "Transition":t["label"], "Source_Stage":t["source_label"],
                    "Target_Stage":t["target_label"], "Case":ci_+1,
                    "Orig_FPG":round(orig_fpg,2), "CF_FPG_mean":round(float(np.mean(pred)),2),
                    "Proximity_Euclid":prox_e, "Proximity_Maha":prox_m,
                    "Diversity":div, "Feasibility":feas, "Sparsity":spar,
                    "N_eligible":len(sub)})
            except Exception as exc:
                log.warning("    Case %d error: %s", ci_+1, exc)
        all_cf_results[grp][stage] = case_list

        # [ABSENT] record recourse availability for this group × transition
        recourse_records.append({"Group":GROUP_LABELS[grp], "_key":grp,
            "Transition":t["label"], "n_eligible":len(sub), "n_attempted":n_attempted,
            "n_recourse":n_recourse, "n_absent":n_attempted - n_recourse})

quality_df = pd.DataFrame(quality_records)

# ── [ABSENT] Recourse-availability table (core Stage-1 finding) ────────────────
recourse_df = pd.DataFrame(recourse_records)
recourse_df["recourse_rate"] = (recourse_df["n_recourse"] /
    recourse_df["n_attempted"].replace(0, np.nan)).round(3)
recourse_df["absence_rate"] = (recourse_df["n_absent"] /
    recourse_df["n_attempted"].replace(0, np.nan)).round(3)
print("\n── [ABSENT] ★ Recourse availability by group × transition ──")
print(recourse_df.drop(columns=["_key"]).to_string(index=False))
print("   recourse_rate = fraction of representative cases with ANY actionable CF.")
print("   recourse_rate = 0.0  →  structural recourse ABSENCE (no feasible path).")
recourse_df.drop(columns=["_key"]).to_csv(
    TABLE_DIR / "dice_recourse_availability.csv", index=False, encoding="utf-8")
joblib.dump(all_cf_results, ART_DIR / "dice_results.pkl")
quality_df.drop(columns=["_key"]).to_csv(TABLE_DIR / "dice_quality_metrics.csv", index=False, encoding="utf-8")
log.info("Saved dice_results.pkl + dice_quality_metrics.csv (%d records)", len(quality_df))

# ═══════════════════════════════════════════════════════════════════════════════
# 4. QUALITY SUMMARY + [MAHA] Euclid-vs-Maha gap
# ═══════════════════════════════════════════════════════════════════════════════
print("\n── DiCE quality by transition (Euclid vs Mahalanobis proximity) ──")
summ = (quality_df.groupby("Transition")[
        ["Proximity_Euclid","Proximity_Maha","Diversity","Feasibility","Sparsity"]]
        .agg(["mean","std"]).round(4))
print(summ.to_string())

# Independence-assumption distortion: ratio Maha/Euclid per group
maha_gap = (quality_df.groupby("Group")[["Proximity_Euclid","Proximity_Maha"]]
            .mean().round(4))
maha_gap["Maha_over_Euclid"] = (maha_gap["Proximity_Maha"] /
                                maha_gap["Proximity_Euclid"].replace(0,np.nan)).round(2)
print("\n── [MAHA] Independence-assumption distortion (Maha / Euclid) ──")
print(maha_gap.to_string())
maha_gap.to_csv(TABLE_DIR / "dice_maha_vs_euclid.csv", encoding="utf-8")

# ═══════════════════════════════════════════════════════════════════════════════
# 5. [CI] BOOTSTRAP — group proximity + elderly/young ratio (IFG → Normal)
# ═══════════════════════════════════════════════════════════════════════════════
ifg = quality_df[quality_df["Transition"]=="IFG → Normal"]
prox_boot_rows = []
for metric in ["Proximity_Euclid","Proximity_Maha"]:
    for grp_label in [GROUP_LABELS[g] for g in GROUP_CONFIG]:
        vals = ifg.loc[ifg["Group"]==grp_label, metric].values
        if len(vals)==0: continue
        lo, hi = bootstrap_ci(vals)
        prox_boot_rows.append({"Metric":metric, "Group":grp_label, "n_cases":len(vals),
            "mean":round(float(np.mean(vals)),4), "CI_lo":lo, "CI_hi":hi})
prox_boot = pd.DataFrame(prox_boot_rows)
print("\n── [CI] Group proximity with 95% bootstrap CI (IFG → Normal) ──")
print(prox_boot.to_string(index=False))
prox_boot.to_csv(TABLE_DIR / "dice_proximity_bootstrap.csv", index=False, encoding="utf-8")

# Headline ratio test: Elderly Male / Young Male (both metrics), with bootstrap CI
def ratio_ci(elder_vals, young_vals, n_boot=N_BOOT, seed=SEED):
    e = np.asarray(elder_vals, float); y = np.asarray(young_vals, float)
    if len(e)==0 or len(y)==0: return (np.nan, np.nan, np.nan)
    rng = np.random.RandomState(seed); rs=[]
    for _ in range(n_boot):
        rs.append(np.mean(rng.choice(e,len(e),replace=True)) /
                  max(np.mean(rng.choice(y,len(y),replace=True)),1e-9))
    return (round(float(np.mean(e))/max(float(np.mean(y)),1e-9),2),
            round(float(np.percentile(rs,2.5)),2),
            round(float(np.percentile(rs,97.5)),2))

ratio_rows=[]
for metric in ["Proximity_Euclid","Proximity_Maha"]:
    em = ifg.loc[ifg["Group"]=="Elderly Male", metric].values
    ym = ifg.loc[ifg["Group"]=="Young Male",   metric].values
    ef = ifg.loc[ifg["Group"]=="Elderly Female", metric].values
    yf = ifg.loc[ifg["Group"]=="Young Female",   metric].values
    r1 = ratio_ci(em, ym); r2 = ratio_ci(ef, yf)
    ratio_rows.append({"Metric":metric, "Comparison":"Elderly Male / Young Male",
                       "Ratio":r1[0], "CI_lo":r1[1], "CI_hi":r1[2]})
    ratio_rows.append({"Metric":metric, "Comparison":"Elderly Female / Young Female",
                       "Ratio":r2[0], "CI_lo":r2[1], "CI_hi":r2[2]})
ratio_df = pd.DataFrame(ratio_rows)
print("\n── [CI] ★ Headline proximity-ratio with 95% bootstrap CI ──")
print(ratio_df.to_string(index=False))
print("   (If CI_lo comfortably > 1, the elderly recourse-burden gap is robust;")
print("    if CI spans 1 or is very wide, the point estimate is 3-case noise.)")
ratio_df.to_csv(TABLE_DIR / "dice_ratio_bootstrap.csv", index=False, encoding="utf-8")

# ═══════════════════════════════════════════════════════════════════════════════
# 6. LLM-READY STRUCTURED OUTPUT (→ Notebook 04)
# ═══════════════════════════════════════════════════════════════════════════════
llm_records=[]
for grp, gcfg in GROUP_CONFIG.items():
    for stage, t in TRANSITIONS.items():
        for case in all_cf_results.get(grp,{}).get(stage,[]):
            delta = case["delta_df"].mean()
            top5 = delta.abs().nlargest(5)
            changes = {f: round(float(delta[f]),3) for f in top5.index}
            for i, pf in enumerate(case["pred_fpg"]):
                sc = "M" if gcfg["sex_code"]==1.0 else "F"
                ac = {0.0:"Y",1.0:"M",2.0:"S"}[gcfg["age_group"]]
                pid = f"{sc}-{ac}-XX-{case['case_idx']:04d}-CF{i+1}"
                qrow = quality_df[(quality_df["Group"]==GROUP_LABELS[grp]) &
                                  (quality_df["Transition"]==t["label"]) &
                                  (quality_df["Case"]==case["case_idx"])]
                llm_records.append({"pseudo_id":pid, "group":GROUP_LABELS[grp],
                    "source_stage":t["source_label"], "target_stage":t["target_label"],
                    "case_idx":case["case_idx"], "cf_idx":i+1,
                    "orig_fpg":round(case["orig_fpg"],2), "pred_fpg":round(float(pf),2),
                    "fpg_reduction":round(case["orig_fpg"]-float(pf),2),
                    "top5_changes":str(changes),
                    "proximity_euclid":qrow["Proximity_Euclid"].values[0] if len(qrow) else None,
                    "proximity_maha":qrow["Proximity_Maha"].values[0] if len(qrow) else None})
llm_df = pd.DataFrame(llm_records)
llm_df.to_csv(ART_DIR / "dice_llm_input.csv", index=False, encoding="utf-8")
log.info("LLM input saved: %d rows", len(llm_df))

# ═══════════════════════════════════════════════════════════════════════════════
# 7. FIGURES  (seaborn grayscale; no captions; dpi=600; png + pdf)
# ═══════════════════════════════════════════════════════════════════════════════
group_order = [GROUP_LABELS[g] for g in GROUP_CONFIG]

# --- Fig 1: proximity by group with bootstrap CI (Euclid) — the key plot ---
if len(prox_boot):
    pe = prox_boot[prox_boot["Metric"]=="Proximity_Euclid"].set_index("Group").reindex(group_order).dropna()
    fig, ax = plt.subplots(figsize=(9,5))
    yerr = np.vstack([pe["mean"]-pe["CI_lo"], pe["CI_hi"]-pe["mean"]])
    ax.barh(pe.index, pe["mean"], color="0.55", edgecolor="black",
            xerr=yerr, capsize=4, error_kw={"ecolor":"0.2","lw":1.2})
    for i,(g,r) in enumerate(pe.iterrows()):
        ax.text(r["mean"]+0.01, i, f"{r['mean']:.3f}", va="center", fontsize=9)
    ax.set_xlabel("Mean proximity, IFG → Normal (Euclidean; 95% bootstrap CI)")
    ax.set_ylabel("")
    plt.tight_layout(); save_fig(fig, "fig_dice_proximity_bootstrap")

# --- Fig 2: Euclid vs Mahalanobis proximity per group ---
mg = maha_gap.reindex(group_order).dropna()
fig, ax = plt.subplots(figsize=(9,5))
x = np.arange(len(mg)); w=0.38
ax.bar(x-w/2, mg["Proximity_Euclid"], w, color="0.30", edgecolor="black", label="Euclidean")
ax.bar(x+w/2, mg["Proximity_Maha"],  w, color="0.75", edgecolor="black", hatch="///", label="Mahalanobis")
ax.set_xticks(x); ax.set_xticklabels(mg.index, rotation=30, ha="right", fontsize=8)
ax.set_ylabel("Mean proximity (IFG → Normal + Diabetes → IFG pooled)")
ax.legend(title="", fontsize=9)
plt.tight_layout(); save_fig(fig, "fig_dice_euclid_vs_maha")

# --- Fig 3: FPG transition flow (grayscale) ---
fig, axes = plt.subplots(2,3, figsize=(15,9)); axes=axes.flatten()
markers={"diabetes":"o","ifg":"s"}
for ax,(grp,gcfg) in zip(axes, GROUP_CONFIG.items()):
    plotted=False
    for stage,mk in markers.items():
        for case in all_cf_results.get(grp,{}).get(stage,[]):
            for pred in case["pred_fpg"]:
                ax.annotate("", xy=(1,pred), xytext=(0,case["orig_fpg"]),
                            arrowprops=dict(arrowstyle="->", color="0.4", lw=1.3, alpha=0.7))
                ax.scatter([0],[case["orig_fpg"]], color="0.15", s=42, zorder=5)
                ax.scatter([1],[pred], color="0.55", s=42, marker=mk, zorder=5,
                           edgecolors="white", lw=0.5)
                plotted=True
    if plotted:
        ax.axhline(100, ls="--", color="0.3", lw=1.1)
        ax.axhline(126, ls=":",  color="0.3", lw=1.1)
    ax.set_title(GROUP_LABELS[grp], fontsize=10, fontweight="bold")
    ax.set_xticks([0,1]); ax.set_xticklabels(["Original","Counterfactual"], fontsize=8)
    ax.set_ylabel("FPG (mg/dL)", fontsize=9)
plt.tight_layout(); save_fig(fig, "fig_dice_fpg_transition")

# --- Fig 4: [ABSENT] recourse availability heatmap (group × transition) ---
if len(recourse_df):
    piv = recourse_df.pivot_table(index="Group", columns="Transition",
                                  values="recourse_rate", aggfunc="first")
    piv = piv.reindex(group_order)
    fig, ax = plt.subplots(figsize=(8, 5))
    sns.heatmap(piv, annot=True, fmt=".2f", cmap="Greys_r", vmin=0, vmax=1,
                linewidths=0.5, linecolor="white",
                cbar_kws={"label":"Recourse availability rate", "shrink":0.8}, ax=ax)
    ax.set_xlabel(""); ax.set_ylabel("")
    plt.tight_layout(); save_fig(fig, "fig_dice_recourse_availability")

log.info("Figures saved (png + pdf, dpi=%d) to results/figures/", DPI)

# ═══════════════════════════════════════════════════════════════════════════════
# 8. SAVE + STAGE-1 VERDICT
# ═══════════════════════════════════════════════════════════════════════════════
joblib.dump({"quality_df":quality_df, "prox_boot":prox_boot, "ratio_df":ratio_df,
             "maha_gap":maha_gap, "recourse_df":recourse_df,
             "all_cf_results":all_cf_results,
             "FEATURES_TO_VARY":FEATURES_TO_VARY, "IMMUTABLE":IMMUTABLE},
            ART_DIR / "dice_diagnostics.pkl")

print("\n" + "="*70)
print("STAGE-1 DIAGNOSTIC VERDICT")
print("="*70)

# (1) Recourse ABSENCE — the core finding
print("\n[1] RECOURSE ABSENCE (actionable features only):")
absent = recourse_df[(recourse_df["n_attempted"] > 0) & (recourse_df["n_recourse"] == 0)]
if len(absent):
    for _, r in absent.iterrows():
        print(f"    ✗ {r['Group']:18s} {r['Transition']:16s} — NO feasible recourse "
              f"({r['n_eligible']} eligible cases, 0 actionable CF)")
    print("    → Stronger than a finite burden: these groups have NO actionable path.")
else:
    print("    (all attempted group×transition cells produced at least one recourse)")

# (2) Proximity-ratio robustness (where recourse exists)
print("\n[2] PROXIMITY-RATIO ROBUSTNESS (where recourse exists):")
if len(ratio_df):
    for _, r in ratio_df.iterrows():
        if np.isnan(r["Ratio"]):
            print(f"    — {r['Metric']:18s} {r['Comparison']:32s} "
                  f"UNDEFINED (recourse absent in one arm)")
        else:
            robust = (not np.isnan(r["CI_lo"])) and (r["CI_lo"] > 1.0)
            print(f"    {'✓' if robust else '△'} {r['Metric']:18s} {r['Comparison']:32s} "
                  f"ratio={r['Ratio']}  CI=[{r['CI_lo']}, {r['CI_hi']}]  "
                  f"{'ROBUST' if robust else 'FRAGILE (CI spans 1)'}")

print("\n[INTERPRETATION]")
print("  The original 'Elderly Male 9.4x' headline does not reproduce: with")
print("  structural variables immutable, Elderly Male IFG→Normal has NO recourse")
print("  (an absence, not a large ratio). Where ratios ARE defined, the Euclidean")
print("  gap shrinks under the correlation-aware Mahalanobis metric — the burden")
print("  is sensitive to (a) immutability, (b) distance metric, (c) sample size.")
log.info("Notebook 03 (Stage-1 diagnostic) complete.")

2026-09-01 15:34:17,065 | INFO | Actionable features: 27 / 31 (immutable: IncomeQuartile, HouseholdIncome, EducationLevel, HealthScreening)
2026-09-01 15:34:17,065 | INFO | Primitives loaded. df_final: (27934, 36)
2026-09-01 15:34:17,070 | INFO | DiCE + proximity + bootstrap utilities defined.
2026-09-01 15:34:17,076 | INFO | ── Group: Young Male ──
2026-09-01 15:34:17,174 | INFO |   [Diabetes (>=126) → IFG (100-125)] eligible=8 representative=3
100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.09it/s]
2026-09-01 15:34:17,714 | INFO |     Case 1 | orig=134.0 ProxE=0.1751 ProxM=19.1833 Div=0.3502 Feas=0.9825 Spar=0.0323
100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.15it/s]
2026-09-01 15:34:18,253 | INFO |     Case 2 | orig=142.0 ProxE=0.2157 ProxM=14.5887 Div=0.3256 Feas=0.9825 Spar=0.0538
100%|███████████████████████████████████████████████████████████████████

No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 04 sec


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:04<00:00,  4.70s/it]
2026-09-01 15:34:45,814 | WARNING | CF generation failed: No counterfactuals found for any of the query points! Kindly check your configuration.
2026-09-01 15:34:45,815 | INFO |     Case 2 | orig=107.0 | NO RECOURSE (no actionable CF found)


No Counterfactuals found for the given configuration, perhaps try with different parameters... ; total time taken: 00 min 04 sec


100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:04<00:00,  4.50s/it]
2026-09-01 15:34:50,352 | WARNING | CF generation failed: No counterfactuals found for any of the query points! Kindly check your configuration.
2026-09-01 15:34:50,352 | INFO |     Case 3 | orig=107.0 | NO RECOURSE (no actionable CF found)
2026-09-01 15:34:50,354 | INFO | ── Group: Elderly Female ──
2026-09-01 15:34:50,460 | INFO |   [Diabetes (>=126) → IFG (100-125)] eligible=130 representative=3


...


── [ABSENT] ★ Recourse availability by group × transition ──
             Group     Transition  n_eligible  n_attempted  n_recourse  n_absent  recourse_rate  absence_rate
        Young Male Diabetes → IFG           8            3           3         0            1.0           0.0
        Young Male   IFG → Normal         125            3           3         0            1.0           0.0
      Young Female Diabetes → IFG           9            3           3         0            1.0           0.0
      Young Female   IFG → Normal          70            3           3         0            1.0           0.0
  Middle-aged Male Diabetes → IFG         137            3           3         0            1.0           0.0
  Middle-aged Male   IFG → Normal         430            3           3         0            1.0           0.0
Middle-aged Female Diabetes → IFG          93            3           3         0            1.0           0.0
Middle-aged Female   IFG → Normal         409            3

2026-09-01 15:34:55,316 | INFO | Saved dice_results.pkl + dice_quality_metrics.csv (33 records)



── DiCE quality by transition (Euclid vs Mahalanobis proximity) ──
               Proximity_Euclid         Proximity_Maha         Diversity         Feasibility         Sparsity        
                           mean     std           mean     std      mean     std        mean     std     mean     std
Transition                                                                                                           
Diabetes → IFG           0.1185  0.0649         9.4838  5.2617    0.2203  0.1261      0.9698  0.0198   0.0478  0.0084
IFG → Normal             0.2341  0.1974        10.5910  4.6821    0.2185  0.0821      0.9661  0.0234   0.0810  0.0645

── [MAHA] Independence-assumption distortion (Maha / Euclid) ──
                    Proximity_Euclid  Proximity_Maha  Maha_over_Euclid
Group                                                                 
Elderly Female                0.2761         11.6957             42.36
Elderly Male                  0.0725          7.9030            

2026-09-01 15:34:57,068 | INFO | LLM input saved: 99 rows
2026-09-01 15:35:08,940 | INFO | Figures saved (png + pdf, dpi=600) to results/figures/
2026-09-01 15:35:09,440 | INFO | Notebook 03 (Stage-1 diagnostic) complete.



STAGE-1 DIAGNOSTIC VERDICT

[1] RECOURSE ABSENCE (actionable features only):
    ✗ Elderly Male       IFG → Normal     — NO feasible recourse (324 eligible cases, 0 actionable CF)
    → Stronger than a finite burden: these groups have NO actionable path.

[2] PROXIMITY-RATIO ROBUSTNESS (where recourse exists):
    — Proximity_Euclid   Elderly Male / Young Male        UNDEFINED (recourse absent in one arm)
    ✓ Proximity_Euclid   Elderly Female / Young Female    ratio=3.02  CI=[2.45, 4.01]  ROBUST
    — Proximity_Maha     Elderly Male / Young Male        UNDEFINED (recourse absent in one arm)
    △ Proximity_Maha     Elderly Female / Young Female    ratio=1.24  CI=[0.9, 1.67]  FRAGILE (CI spans 1)

[INTERPRETATION]
  The original 'Elderly Male 9.4x' headline does not reproduce: with
  structural variables immutable, Elderly Male IFG→Normal has NO recourse
  (an absence, not a large ratio). Where ratios ARE defined, the Euclidean
  gap shrinks under the correlation-aware Mahalanobis me

In [2]:
df_final          = pd.read_parquet(OUTPUT_DIR / "df_final.parquet")
best_models       = joblib.load(OUTPUT_DIR / "best_models.pkl")
best_algo_name    = joblib.load(OUTPUT_DIR / "best_algo_name.pkl")
best_params_store = joblib.load(OUTPUT_DIR / "best_params_store.pkl")

NUM_FEATURES = [
    "BMI", "WaistCirc", "Weight",
    "Energy_kcal", "Carb_g",  "Sugar_g",  "Sodium_mg",
    "Fat_g",       "SatFat_g","Fiber_g",   "Potassium_mg", "Protein_g",
]
CAT_FEATURES = [
    "ObesityStatus",    "WeightChangeStatus", "WeightLossAmount", "WeightGainAmount",
    "DrinkingFrequency","DrinkingAmount",      "SmokingStatus",
    "VigorousAct_Work", "VigorousAct_Leisure","ModerateAct_Work",
    "WalkingActivity",  "AerobicRate",         "BreakfastFreq",
    "StressLevel",      "StressAwareness",
    "IncomeQuartile",   "HouseholdIncome",     "EducationLevel", "HealthScreening",
]
X_FEATURES = NUM_FEATURES + CAT_FEATURES

log.info("Artefacts loaded. df_final: %s", df_final.shape)


2026-04-27 15:14:42,293 | INFO | Artefacts loaded. df_final: (16677, 36)


## 3. DiCE Helper Functions

In [3]:
def force_float(df: pd.DataFrame, features: list) -> pd.DataFrame:
    """Cast all feature columns to float64 (required by DiCE)."""
    df = df.copy()
    for col in features:
        df[col] = pd.to_numeric(df[col], errors="coerce").astype("float64")
    return df


def build_dice_explainer(model, df_train: pd.DataFrame,
                         feature_names: list, outcome_name: str = "FPG"):
    """
    Initialise DiCE Data + Model objects for regression.
    df_train must have float64 features and the outcome column.
    """
    d   = dice_ml.Data(
        dataframe           = df_train[feature_names + [outcome_name]],
        continuous_features = feature_names,
        outcome_name        = outcome_name,
    )
    m   = dice_ml.Model(model=model, backend="sklearn", model_type="regressor")
    exp = Dice(d, m, method="random")
    return exp


def generate_counterfactuals(explainer, query: pd.DataFrame,
                             target_range: tuple, n_cf: int = 3):
    """
    Generate n_cf counterfactual instances for a single query.

    Parameters
    ----------
    query        : float64 DataFrame, shape (1, n_features)
    target_range : (low, high) desired FPG range
    Returns None on failure.
    """
    query = force_float(query, query.columns.tolist())
    try:
        result = explainer.generate_counterfactuals(
            query_instances  = query,
            total_CFs        = n_cf,
            desired_range    = list(target_range),
            permitted_range  = None,
            features_to_vary = "all",
            random_seed      = SEED,
        )
        return result
    except Exception as exc:
        log.warning("CF generation failed: %s", exc)
        return None


# ── Quality metrics ───────────────────────────────────────────────────────────
def _feature_stds(df_train: pd.DataFrame, features: list) -> np.ndarray:
    """Per-feature standard deviations from training data (used for normalisation)."""
    stds = df_train[features].std().values
    stds[stds == 0] = 1.0
    return stds


def proximity(original: pd.DataFrame, cf_df: pd.DataFrame,
              features: list, stds: np.ndarray) -> float:
    """
    Mean normalised L1 distance from original to each counterfactual.
    Lower is better — indicates minimal lifestyle change required.
    """
    orig   = original[features].values.flatten().astype(float)
    scores = [float(np.mean(np.abs(orig - row.values.astype(float)) / stds))
              for _, row in cf_df[features].iterrows()]
    return round(float(np.mean(scores)), 4)


def diversity(cf_df: pd.DataFrame, features: list, stds: np.ndarray) -> float:
    """
    Mean pairwise normalised L1 distance among counterfactuals.
    Higher is better — more diverse action pathways offered.
    """
    if len(cf_df) < 2:
        return 0.0
    vals  = cf_df[features].values.astype(float)
    dists = [float(np.mean(np.abs(vals[i] - vals[j]) / stds))
             for i in range(len(vals)) for j in range(i+1, len(vals))]
    return round(float(np.mean(dists)), 4)


def feasibility(original: pd.DataFrame, cf_df: pd.DataFrame,
                features: list, cat_features: list) -> float:
    """
    Proportion of categorical feature changes ≤ 1 unit.
    Higher is better — changes must be clinically realistic.
    """
    orig_cats = original[cat_features].values.flatten().astype(float)
    n_total   = len(cf_df) * len(cat_features)
    if n_total == 0:
        return 0.0
    n_valid = sum(
        int(abs(cv - ov) <= 1.0)
        for _, row in cf_df[cat_features].iterrows()
        for ov, cv in zip(orig_cats, row.values.astype(float))
    )
    return round(n_valid / n_total, 4)


def sparsity(original: pd.DataFrame, cf_df: pd.DataFrame,
             features: list, tol: float = 1e-3) -> float:
    """
    Mean proportion of features changed relative to the original.
    Lower is better — fewer changes → more actionable recommendation.
    """
    orig   = original[features].values.flatten().astype(float)
    scores = [np.sum(np.abs(orig - row.values.astype(float)) > tol) / len(features)
              for _, row in cf_df[features].iterrows()]
    return round(float(np.mean(scores)), 4)


log.info("DiCE utilities defined.")


2026-04-27 15:14:42,343 | INFO | DiCE utilities defined.


## 4. Stratified DiCE Generation

In [4]:
log.info("=" * 60)
log.info("DiCE counterfactual generation — 6 groups × 2 transitions")
log.info("=" * 60)

all_cf_results  = {}   # {group: {transition: [case_dicts]}}
quality_records = []   # flat list for quality DataFrame

for grp, cfg in GROUP_CONFIG.items():
    log.info("── Group: %s ──", GROUP_LABELS[grp])
    all_cf_results[grp] = {}

    df_g = df_final[
        (df_final["AgeGroup"] == cfg["age_group"]) &
        (df_final["Sex"]      == cfg["sex_code"])
    ].copy().reset_index(drop=True)

    df_g = force_float(df_g, X_FEATURES)
    df_g["FPG"] = pd.to_numeric(df_g["FPG"], errors="coerce").astype("float64")

    model = best_models.get(grp)
    if model is None:
        log.warning("  No model for %s — skipping", grp)
        continue

    X = df_g[X_FEATURES]
    y = df_g["FPG"]
    X_train, X_val, y_train, y_val = train_test_split(
        X, y, test_size=0.20, random_state=SEED)

    df_train = force_float(X_train.copy(), X_FEATURES)
    df_train["FPG"] = y_train.values.astype("float64")
    stds = _feature_stds(df_train, X_FEATURES)

    try:
        explainer = build_dice_explainer(model, df_train, X_FEATURES)
    except Exception as exc:
        log.error("  DiCE explainer failed for %s: %s", grp, exc)
        continue

    df_val = force_float(X_val.copy(), X_FEATURES)
    df_val["FPG"]   = y_val.values.astype("float64")
    df_val["Stage"] = df_val["FPG"].apply(assign_glucose_stage)

    for stage, t_info in TRANSITIONS.items():
        target_range = t_info["target_range"]
        sub = df_val[df_val["Stage"] == stage].reset_index(drop=True)
        if len(sub) == 0:
            log.info("  [%s] No cases in hold-out for stage '%s'", grp, stage)
            continue

        # Select N_CASES cases closest to the group median FPG
        median_fpg  = sub["FPG"].median()
        rep_cases   = sub.iloc[(sub["FPG"] - median_fpg).abs().argsort()].head(N_CASES)
        n_available = len(sub)
        log.info("  [%s → %s]  eligible=%d  representative=%d",
                 t_info["source_label"], t_info["target_label"],
                 n_available, len(rep_cases))

        case_list = []
        for case_idx, (_, row) in enumerate(rep_cases.iterrows()):
            query     = force_float(row[X_FEATURES].to_frame().T.reset_index(drop=True),
                                    X_FEATURES)
            orig_fpg  = float(row["FPG"])

            cf_result = generate_counterfactuals(explainer, query,
                                                 target_range, n_cf=N_CF)
            if cf_result is None:
                continue
            try:
                cf_df = cf_result.cf_examples_list[0].final_cfs_df
                if cf_df is None or len(cf_df) == 0:
                    continue
                cf_df    = force_float(cf_df, X_FEATURES)
                delta_df = cf_df[X_FEATURES].subtract(query[X_FEATURES].values[0],
                                                       axis=1)
                prox = proximity(query, cf_df, X_FEATURES, stds)
                div  = diversity(cf_df, X_FEATURES, stds)
                feas = feasibility(query, cf_df, X_FEATURES, CAT_FEATURES)
                spar = sparsity(query, cf_df, X_FEATURES)
                pred_fpg = model.predict(cf_df[X_FEATURES]).tolist()

                log.info("    Case %d | orig=%.1f  Prox=%.4f  Div=%.4f  "
                         "Feas=%.4f  Spar=%.4f | CF preds: %s",
                         case_idx+1, orig_fpg, prox, div, feas, spar,
                         [f"{p:.1f}" for p in pred_fpg])

                case_list.append({
                    "case_idx": case_idx + 1,
                    "orig_fpg": orig_fpg,
                    "pred_fpg": pred_fpg,
                    "cf_df":    cf_df,
                    "delta_df": delta_df,
                    "query":    query,
                    "n_eligible": n_available,
                })
                quality_records.append({
                    "Group":        GROUP_LABELS[grp],
                    "Transition":   t_info["label"],
                    "Source_Stage": t_info["source_label"],
                    "Target_Stage": t_info["target_label"],
                    "Case":         case_idx + 1,
                    "Orig_FPG":     round(orig_fpg, 2),
                    "CF_FPG_mean":  round(float(np.mean(pred_fpg)), 2),
                    "Proximity":    prox,
                    "Diversity":    div,
                    "Feasibility":  feas,
                    "Sparsity":     spar,
                    "N_eligible":   n_available,
                })
            except Exception as exc:
                log.warning("    Case %d processing error: %s", case_idx+1, exc)

        all_cf_results[grp][stage] = case_list

# Save
joblib.dump(all_cf_results, OUTPUT_DIR / "dice_results.pkl")
quality_df = pd.DataFrame(quality_records)
quality_df.to_csv(OUTPUT_DIR / "dice_quality_metrics.csv", index=False, encoding="utf-8")
log.info("Saved: dice_results.pkl + dice_quality_metrics.csv (%d records)", len(quality_df))


2026-04-27 15:14:42,385 | INFO | ============================================================
2026-04-27 15:14:42,387 | INFO | DiCE counterfactual generation — 6 groups × 2 transitions
2026-04-27 15:14:42,394 | INFO | ============================================================
2026-04-27 15:14:42,398 | INFO | ── Group: Young Male ──
2026-04-27 15:14:42,563 | INFO |   [Diabetes (≥126) → IFG (100–125)]  eligible=12  representative=3
100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.35it/s]
2026-04-27 15:14:43,069 | INFO |     Case 1 | orig=133.0  Prox=0.1548  Div=0.3095  Feas=0.9649  Spar=0.0538 | CF preds: ['108.5', '102.6', '103.7']
100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00,  2.50it/s]
2026-04-27 15:14:43,561 | INFO |     Case 2 | orig=144.0  Prox=0.1463  Div=0.1623  Feas=0.9649  Spar=0.0538 | CF preds: ['100.7', '103.1', '100.7']
100%|███████████████████████

## 5. Quality Metrics Summary

In [5]:
print("── DiCE Quality Metrics Summary ──")
summary = (quality_df
    .groupby(["Transition"])[["Proximity","Diversity","Feasibility","Sparsity"]]
    .agg(["mean","std"])
    .round(4))
print(summary.to_string())

print("\n── Per-Group: IFG → Normal Proximity (key fairness indicator) ──")
prox_ifg = (quality_df[quality_df["Transition"] == "IFG → Normal"]
            .groupby("Group")["Proximity"]
            .agg(["mean","min","max"])
            .round(4))
print(prox_ifg.sort_values("mean", ascending=False).to_string())

print("\n[Metric Interpretation]")
for m, desc in [
    ("Proximity",    "↓ Lower = smaller feature change needed (easier to act on)"),
    ("Diversity",    "↑ Higher = more varied action pathways offered"),
    ("Feasibility",  "↑ Higher = changes within realistic bounds"),
    ("Sparsity",     "↓ Lower = fewer features need to change"),
]:
    print(f"  {m:12s}: {desc}")


── DiCE Quality Metrics Summary ──
               Proximity         Diversity         Feasibility         Sparsity        
                    mean     std      mean     std        mean     std     mean     std
Transition                                                                             
Diabetes → IFG    0.1149  0.0477    0.1937  0.0895      0.9737  0.0183   0.0532  0.0101
IFG → Normal      0.3620  0.3600    0.2773  0.2024      0.9259  0.0793   0.1493  0.1278

── Per-Group: IFG → Normal Proximity (key fairness indicator) ──
                      mean     min     max
Group                                     
Elderly Male        0.9735  0.6872  1.3746
Elderly Female      0.4682  0.1393  0.7765
Middle-aged Male    0.3809  0.2404  0.4522
Young Female        0.1430  0.0977  0.2079
Middle-aged Female  0.1033  0.0759  0.1413
Young Male          0.1029  0.0806  0.1420

[Metric Interpretation]
  Proximity   : ↓ Lower = smaller feature change needed (easier to act on)
  Diversity   :

## 6. LLM-Ready Structured Output (→ Notebook 04)

In [6]:
llm_records = []

for grp, cfg in GROUP_CONFIG.items():
    for stage, t_info in TRANSITIONS.items():
        case_list = all_cf_results.get(grp, {}).get(stage, [])
        for case in case_list:
            delta     = case["delta_df"].mean()
            top5_feat = delta.abs().nlargest(5)
            changes   = {f: round(float(delta[f]), 3) for f in top5_feat.index}

            for i, pred_fpg in enumerate(case["pred_fpg"]):
                # Pseudonymised ID encoding sex (M/F), age group (Y/M/S), stage, case, CF
                sex_code  = "M" if cfg["sex_code"] == 1.0 else "F"
                age_code  = {0.0:"Y", 1.0:"M", 2.0:"S"}[cfg["age_group"]]
                pseudo_id = (f"{sex_code}-{age_code}-XX-"
                             f"{case['case_idx']:04d}-CF{i+1}")

                llm_records.append({
                    "pseudo_id":      pseudo_id,
                    "group":          GROUP_LABELS[grp],
                    "source_stage":   t_info["source_label"],
                    "target_stage":   t_info["target_label"],
                    "case_idx":       case["case_idx"],
                    "cf_idx":         i + 1,
                    "orig_fpg":       round(case["orig_fpg"], 2),
                    "pred_fpg":       round(float(pred_fpg), 2),
                    "fpg_reduction":  round(case["orig_fpg"] - float(pred_fpg), 2),
                    "top5_changes":   str(changes),
                    "proximity":      quality_df.loc[
                        (quality_df["Group"]      == GROUP_LABELS[grp]) &
                        (quality_df["Transition"] == t_info["label"]) &
                        (quality_df["Case"]       == case["case_idx"]),
                        "Proximity"].values[0] if len(quality_df) > 0 else None,
                })

llm_df = pd.DataFrame(llm_records)
llm_df.to_csv(OUTPUT_DIR / "dice_llm_input.csv", index=False, encoding="utf-8")
log.info("LLM input saved: %d rows × %d cols", *llm_df.shape)
print(llm_df[["pseudo_id","group","source_stage","orig_fpg","pred_fpg","top5_changes"]]
      .head(9).to_string(index=False))


2026-04-27 15:15:11,677 | INFO | LLM input saved: 108 rows × 11 cols


      pseudo_id      group    source_stage  orig_fpg  pred_fpg                                                                                                        top5_changes
M-Y-XX-0001-CF1 Young Male Diabetes (≥126)     133.0    108.49 {'Sugar_g': 67.115, 'Fiber_g': 25.46, 'DrinkingAmount': 0.667, 'HouseholdIncome': 0.567, 'VigorousAct_Work': 0.333}
M-Y-XX-0001-CF2 Young Male Diabetes (≥126)     133.0    102.58 {'Sugar_g': 67.115, 'Fiber_g': 25.46, 'DrinkingAmount': 0.667, 'HouseholdIncome': 0.567, 'VigorousAct_Work': 0.333}
M-Y-XX-0001-CF3 Young Male Diabetes (≥126)     133.0    103.68 {'Sugar_g': 67.115, 'Fiber_g': 25.46, 'DrinkingAmount': 0.667, 'HouseholdIncome': 0.567, 'VigorousAct_Work': 0.333}
M-Y-XX-0002-CF1 Young Male Diabetes (≥126)     144.0    100.69                     {'Potassium_mg': 4343.256, 'WaistCirc': 5.1, 'ObesityStatus': 1.033, 'BMI': 0.0, 'Weight': 0.0}
M-Y-XX-0002-CF2 Young Male Diabetes (≥126)     144.0    103.08                     {'Potassium_mg': 4343.

## 7. Figures

In [7]:
# ── Figure: Mean feature changes by transition ───────────────────────────────
TOP_N = 10
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for ax, (stage, t_info) in zip(axes, TRANSITIONS.items()):
    delta_all = []
    for grp in GROUP_CONFIG:
        case_list = all_cf_results.get(grp, {}).get(stage, [])
        for case in case_list:
            delta_all.append(case["delta_df"].mean())

    if not delta_all:
        ax.set_title(f"{t_info['label']} — No data")
        continue

    delta_mean = pd.concat(delta_all, axis=1).mean(axis=1)
    top_idx    = delta_mean.abs().nlargest(TOP_N).index
    top_vals   = delta_mean[top_idx]

    bar_colors = ["#E57373" if v > 0 else "#64B5F6" for v in top_vals]
    bars = ax.barh(top_idx, top_vals, color=bar_colors, alpha=0.85,
                   edgecolor="white", linewidth=0.6)

    for bar, val in zip(bars, top_vals):
        ax.text(val + (0.003 if val >= 0 else -0.003),
                bar.get_y() + bar.get_height()/2,
                f"{val:+.3f}", va="center", fontsize=8.5,
                ha="left" if val >= 0 else "right")

    ax.axvline(0, color="black", lw=0.8)
    ax.set_title(f"{t_info['label']}\nMean feature change (top {TOP_N})",
                 fontsize=11, fontweight="bold")
    ax.set_xlabel("Feature change magnitude (original scale)", fontsize=10)
    ax.invert_yaxis()
    ax.spines[["top","right"]].set_visible(False)

    # Legend
    ax.legend(handles=[
        mpatches.Patch(color="#E57373", alpha=0.85, label="Increase"),
        mpatches.Patch(color="#64B5F6", alpha=0.85, label="Decrease"),
    ], fontsize=9, loc="lower right")

plt.suptitle("DiCE Counterfactual — Mean Feature Changes by Transition Pathway",
             fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "fig_dice_feature_changes.png", dpi=DPI, bbox_inches="tight")
plt.show()
log.info("Feature change figure saved.")


2026-04-27 15:15:12,583 | INFO | Feature change figure saved.


In [8]:
# ── Figure: DiCE quality radar by group ──────────────────────────────────────
if len(quality_df) == 0:
    log.warning("No quality records — skipping radar chart")
else:
    categories    = ["Proximity\n(inverted)", "Diversity",
                     "Feasibility", "Sparsity\n(inverted)"]
    N             = len(categories)
    angles        = np.linspace(0, 2*np.pi, N, endpoint=False).tolist() + [0]

    fig, axes = plt.subplots(2, 3, figsize=(15, 10),
                              subplot_kw=dict(polar=True))
    axes = axes.flatten()

    for ax, (grp, cfg) in zip(axes, GROUP_CONFIG.items()):
        sub = quality_df[quality_df["Group"] == GROUP_LABELS[grp]]
        color = GROUP_COLORS[grp]
        if len(sub) == 0:
            ax.set_title(GROUP_LABELS[grp], fontsize=10)
            ax.axis("off")
            continue

        prox_raw = sub["Proximity"].mean()
        # Invert proximity and sparsity: lower raw → higher radar score
        s_prox = max(0.0, 1.0 - min(prox_raw, 1.0))
        s_div  = min(sub["Diversity"].mean(), 1.0)
        s_feas = sub["Feasibility"].mean()
        s_spar = max(0.0, 1.0 - sub["Sparsity"].mean())
        scores = [s_prox, s_div, s_feas, s_spar]
        scores_closed = scores + [scores[0]]

        ax.plot(angles, scores_closed, "o-", color=color, lw=2.2, ms=6)
        ax.fill(angles, scores_closed, color=color, alpha=0.20)
        ax.plot(angles, [0.7]*5, "--", color="gray", lw=0.8, alpha=0.5)

        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(categories, fontsize=8.5)
        ax.set_ylim(0, 1)
        ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
        ax.set_yticklabels(["0.2","0.4","0.6","0.8","1.0"],
                            fontsize=7, color="gray")

        for angle, score in zip(angles[:-1], scores):
            ax.text(angle, score + 0.09, f"{score:.2f}",
                    ha="center", fontsize=8.5, color=color, fontweight="bold",
                    bbox=dict(boxstyle="round,pad=0.15", fc="white",
                              ec=color, alpha=0.8))
        ax.set_title(GROUP_LABELS[grp], fontsize=10, fontweight="bold", pad=10)

    plt.suptitle("DiCE Counterfactual Quality by Demographic Group\n"
                 "(higher score = better on each axis)",
                 fontsize=12, fontweight="bold")
    plt.tight_layout()
    fig.savefig(OUTPUT_DIR / "fig_dice_quality_radar.png", dpi=DPI, bbox_inches="tight")
    plt.show()
    log.info("Quality radar saved.")


2026-04-27 15:15:14,734 | INFO | Quality radar saved.


In [9]:
# ── Figure: FPG transition flow (original → counterfactual) ─────────────────
fig, axes = plt.subplots(2, 3, figsize=(15, 9), sharey=False)
axes      = axes.flatten()

stage_markers = {"diabetes": "o", "ifg": "s"}
stage_labels  = {"diabetes": "Diabetes→IFG", "ifg": "IFG→Normal"}

for ax, (grp, cfg) in zip(axes, GROUP_CONFIG.items()):
    color   = GROUP_COLORS[grp]
    plotted = False

    for stage, marker in stage_markers.items():
        case_list = all_cf_results.get(grp, {}).get(stage, [])
        for case in case_list:
            orig  = case["orig_fpg"]
            preds = case["pred_fpg"]
            for pred in preds:
                ax.annotate("",
                    xy=(1, pred), xytext=(0, orig),
                    arrowprops=dict(arrowstyle="->", color=color,
                                    lw=1.4, alpha=0.65))
                ax.scatter([0], [orig], color="gray",    s=45, zorder=5)
                ax.scatter([1], [pred], color=color, s=45,
                           marker=marker, zorder=5, edgecolors="white", lw=0.5)
                plotted = True

    if plotted:
        ax.axhline(100, color="#FF9800", linestyle="--",
                   lw=1.2, alpha=0.75, label="IFG boundary (100)")
        ax.axhline(126, color="#F44336", linestyle="--",
                   lw=1.2, alpha=0.75, label="Diabetes boundary (126)")
        ax.set_xticks([0, 1])
        ax.set_xticklabels(["Original FPG", "Counterfactual FPG"], fontsize=9)
        ax.set_ylabel("Fasting Plasma Glucose (mg/dL)", fontsize=9)
        ax.legend(fontsize=7.5, loc="upper right")

    ax.set_title(GROUP_LABELS[grp], fontsize=10, fontweight="bold")
    ax.spines[["top","right"]].set_visible(False)

# Shared marker legend
legend_handles = [
    mpatches.Patch(color="gray",    label="● Original"),
    mpatches.Patch(color="#888888", label="● Diabetes → IFG"),
    mpatches.Patch(color="#888888", label="■ IFG → Normal"),
]
fig.legend(
    handles=[
        plt.Line2D([0],[0], marker="o", color="w", markerfacecolor="gray",
                   markersize=9, label="Original FPG"),
        plt.Line2D([0],[0], marker="o", color="w", markerfacecolor="#555",
                   markersize=9, label="CF: Diabetes→IFG"),
        plt.Line2D([0],[0], marker="s", color="w", markerfacecolor="#555",
                   markersize=9, label="CF: IFG→Normal"),
    ],
    loc="lower center", ncol=3, fontsize=9,
    bbox_to_anchor=(0.5, -0.03),
)
plt.suptitle("DiCE Counterfactual — FPG Stage Transition Flow by Group",
             fontsize=12, fontweight="bold", y=1.01)
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "fig_dice_fpg_transition.png", dpi=DPI, bbox_inches="tight")
plt.show()
log.info("FPG transition figure saved.")


2026-04-27 15:15:18,464 | INFO | FPG transition figure saved.


In [10]:
# ── Figure: Proximity by group (key fairness insight) ────────────────────────
# Shows that elderly groups require much larger feature changes → structural inequity

fig, ax = plt.subplots(figsize=(10, 5))

df_prox = (quality_df[quality_df["Transition"] == "IFG → Normal"]
           .groupby("Group")["Proximity"]
           .agg(["mean","std"])
           .reset_index())

# Sort by mean proximity descending
df_prox = df_prox.sort_values("mean", ascending=True)

bar_colors = [GROUP_COLORS.get(
    {v:k for k,v in GROUP_LABELS.items()}.get(g, ""), "#888888")
    for g in df_prox["Group"]]

bars = ax.barh(df_prox["Group"], df_prox["mean"], color=bar_colors,
               alpha=0.85, edgecolor="white", linewidth=0.6,
               xerr=df_prox["std"], capsize=4, error_kw={"ecolor":"gray","lw":1.2})

for bar, val in zip(bars, df_prox["mean"]):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=9.5)

ax.set_xlabel("Mean Proximity (IFG → Normal transition)\n"
              "(lower = smaller lifestyle change needed)",
              fontsize=10)
ax.set_title("Counterfactual Difficulty by Demographic Group\n"
             "(Elderly groups require 2–10× larger changes than Young Male)",
             fontsize=11, fontweight="bold")
ax.axvline(df_prox["mean"].mean(), color="#C62828", linestyle="--",
           lw=1.5, alpha=0.7, label=f"Mean = {df_prox['mean'].mean():.3f}")
ax.legend(fontsize=9)
ax.spines[["top","right"]].set_visible(False)

plt.tight_layout()
fig.savefig(OUTPUT_DIR / "fig_dice_proximity_comparison.png",
            dpi=DPI, bbox_inches="tight")
plt.show()
log.info("Proximity comparison figure saved.")


2026-04-27 15:15:18,942 | INFO | Proximity comparison figure saved.


## 8. Summary & Outputs

In [11]:
log.info("=" * 55)
log.info("Notebook 03 complete.")
log.info("=" * 55)

outputs = [
    ("dice_results.pkl",            "All CF results — loaded by Notebook 04"),
    ("dice_quality_metrics.csv",    "Per-case quality metrics (Proximity/Diversity/Feasibility/Sparsity)"),
    ("dice_llm_input.csv",          "Structured LLM prompt input — loaded by Notebook 04"),
    ("fig_dice_feature_changes.png","Figure: Mean feature changes by transition"),
    ("fig_dice_quality_radar.png",  "Figure: Quality radar by group"),
    ("fig_dice_fpg_transition.png", "Figure: FPG stage transition flow"),
    ("fig_dice_proximity_comparison.png","Figure: Proximity comparison (fairness insight)"),
]
for fname, desc in outputs:
    log.info("  %-42s %s", fname, desc)

print("\n── Final Quality Summary ──")
print(quality_df.groupby("Transition")[
    ["Proximity","Diversity","Feasibility","Sparsity"]
].mean().round(4).to_string())


2026-04-27 15:15:18,971 | INFO | =======================================================
2026-04-27 15:15:18,976 | INFO | Notebook 03 complete.
2026-04-27 15:15:18,980 | INFO | =======================================================
2026-04-27 15:15:18,984 | INFO |   dice_results.pkl                           All CF results — loaded by Notebook 04
2026-04-27 15:15:18,985 | INFO |   dice_quality_metrics.csv                   Per-case quality metrics (Proximity/Diversity/Feasibility/Sparsity)
2026-04-27 15:15:18,985 | INFO |   dice_llm_input.csv                         Structured LLM prompt input — loaded by Notebook 04
2026-04-27 15:15:18,986 | INFO |   fig_dice_feature_changes.png               Figure: Mean feature changes by transition
2026-04-27 15:15:18,988 | INFO |   fig_dice_quality_radar.png                 Figure: Quality radar by group
2026-04-27 15:15:18,989 | INFO |   fig_dice_fpg_transition.png                Figure: FPG stage transition flow
2026-04-27 15:15:18,990 | INFO |


── Final Quality Summary ──
                Proximity  Diversity  Feasibility  Sparsity
Transition                                                 
Diabetes → IFG     0.1149     0.1937       0.9737    0.0532
IFG → Normal       0.3620     0.2773       0.9259    0.1493
